# 03 - Biennial growth rhythmicity and its environmental drivers

The results generated here were used in manuscript sections 3.4 and 3.5.
Ninety principal axes were followed monthly
for 25 months. This notebook first visualises the biennial growth/rest rhythm,
then asks which environmental signals govern each of the five morphogenetic
traits.

The regression uses 16 predictors: the 13 environmental features plus three
contextual flags - rhythm phase, cultivation system and sex. Including the
flags is deliberate: it lets the model tell us how much of each trait the
*environment* explains once the endogenous rhythm and the design are accounted
for, and how much weight sex carries on trait intensity.

**Produces:** Figure 5, Supplementary Figure 2.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from yerbamate import config as C
from yerbamate import models as M, plotting as P

P.use_paper_style()
pd.set_option("display.width", 170, "display.max_columns", 30)

df = pd.read_csv(C.UNIFIED)
print(f"{len(df):,} axis-month records, {df.axis_id.nunique()} axes, {df.time_idx.nunique()} months")
print(df.groupby("time_idx").phase_label.first().value_counts().to_string())

## Supplementary Figure 2 - cumulative growth trajectories

Every tagged axis drawn individually, females in red and males in blue, with
the growth (red) and rest (blue) periods shaded behind. Left column is
monoculture, right column agroforestry.

Shoot elongation shows the sharpest phase transitions, clearest in
monoculture where individual axes range from about 50 to 260 cm over the two
years. Leaf number and leaf area track that seasonality closely in monoculture
but are attenuated under the agroforestry canopy.

In [ ]:
PLOT_VARS = ["elongation", "leaf_increase", "leaf_area_increase", "leaf_shed"]
YLABELS = ["Cumulative shoot\nelongation (cm)", "Cumulative leaf\nnumber increase",
           "Cumulative leaf\narea increase (cm²)", "Cumulative\nleaf shed"]
phase = df.groupby("time_idx")["is_cresc"].first()
xticks = list(range(0, 25, 3))

fig, axes = plt.subplots(4, 2, figsize=(18, 17))
for row, (v, ylab) in enumerate(zip(PLOT_VARS, YLABELS)):
    for col, env in enumerate(["MO", "FUS"]):
        ax = axes[row, col]
        sub = df[df.environment == env].sort_values(["axis_id", "time_idx"])
        for aid, axis_data in sub.groupby("axis_id"):
            d = axis_data.sort_values("time_idx")
            colour = P.COL_FEMALE if d.sex.iloc[0] == "F" else P.COL_MALE
            ax.plot(d.time_idx, np.maximum(d[v], 0).cumsum(), "-",
                    color=colour, alpha=0.45, lw=1.1)
        for t, growing in phase.items():
            ax.axvspan(t - 0.5, t + 0.5, alpha=0.06, lw=0,
                       color="#e74c3c" if growing else "#3498db")
        ax.set_xlim(-0.5, 24.5)
        ax.set_xticks(xticks)
        ax.set_xticklabels([C.DATE_LABELS[i] for i in xticks] if row == 3 else [],
                           rotation=40, ha="right", fontsize=16)
        ax.tick_params(axis="y", labelsize=17)
        ax.set_ylabel(ylab, fontsize=19)
        P.despine(ax)
        P.panel(ax, "ABCDEFGH"[row * 2 + col], x=0.0, y=1.02, size=22)
        if row == 0 and col == 0:
            ax.legend(handles=[Line2D([0], [0], color=P.COL_FEMALE, label="Female", lw=3),
                               Line2D([0], [0], color=P.COL_MALE, label="Male", lw=3),
                               mpatches.Patch(color="#e74c3c", alpha=0.25, label="Growth"),
                               mpatches.Patch(color="#3498db", alpha=0.25, label="Rest")],
                      loc="upper left", frameon=False, fontsize=17)
fig.tight_layout(w_pad=4.0, h_pad=1.6)
P.save(fig, "Figure_S2")

## Figure 5 - driver hierarchy for the five morphogenetic traits

One GBM regressor per trait on the 16 predictors, scored by 5-fold CV and
ranked by permutation importance. Only the ten strongest predictors are drawn.

The models explain roughly a third to a half of the month-to-month variation
in the four size-related traits; leaf shed is the exception at CV R² ~ 0.09.
The unexplained share is genuine axis-to-axis individuality, not model failure
- the predictors are identical for every axis within a system and month, so
nothing in the feature set can distinguish one axis from its neighbour.

In [ ]:
REG_FEATURES = C.GROWTH_REG_FEATURES
print(f"{len(REG_FEATURES)} predictors: {len(C.ENV_FEATURES)} environmental + 3 contextual flags")
print(REG_FEATURES)

r2_rows, importance = [], {}
for v in C.MORPHO_VARS:
    cv_r2, imp = M.gbm_regression(df, REG_FEATURES, v)
    importance[v] = imp.round(1)
    r2_rows.append({"Trait": v, "CV_R2": cv_r2})

r2 = pd.DataFrame(r2_rows).set_index("Trait")["CV_R2"]
pd.DataFrame({"Trait": r2.index, "CV_R2": r2.values}).to_csv(
    C.TAB_DIR / "Figure_5_growth_CV_R2.csv", index=False)

fi_long = pd.DataFrame(
    [{"Trait": v, "Feature": f, "Label": C.FEATURE_LABELS[f],
      "Category": C.FEATURE_CATEGORY[f], "Importance_pct": importance[v][f]}
     for v in C.MORPHO_VARS for f in REG_FEATURES])
fi_long.sort_values(["Trait", "Importance_pct"], ascending=[True, False]).to_csv(
    C.TAB_DIR / "Figure_5_growth_importance.csv", index=False)

print(r2.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.ravel()
for i, v in enumerate(C.MORPHO_VARS):
    ax = axes[i]
    top = importance[v].sort_values(ascending=True).tail(10)
    ax.barh(range(len(top)), top.values, alpha=0.92, edgecolor="white",
            color=[P.CATEGORY_COLORS[C.FEATURE_CATEGORY[f]] for f in top.index])
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([C.FEATURE_LABELS[f] for f in top.index], fontsize=12)
    for j, val in enumerate(top.values):
        if val > 0.5:
            ax.text(val + 0.4, j, f"{val:.1f}", va="center", fontsize=11)
    ax.set_xlabel("Importance (%)")
    ax.set_xlim(0, max(top.values) * 1.2)
    ax.set_title(C.MORPHO_LABELS[v], fontsize=14.5, fontweight="bold", pad=16)
    ax.annotate(f"CV R² = {r2[v]:.2f}", xy=(0.96, 0.06), xycoords="axes fraction",
                ha="right", fontsize=12.5,
                bbox=dict(boxstyle="round", fc="white", ec="#999", alpha=0.85))
    P.despine(ax)
    P.panel(ax, "ABCDE"[i], x=-0.02, y=1.06, size=20)

axes[5].axis("off")
axes[5].legend(handles=[mpatches.Patch(color=P.CATEGORY_COLORS[c], label=c)
                        for c in ["Light (midday)", "Light (low-angle)", "Thermal",
                                  "Photoperiod", "Water", "Sex", "Phase", "System"]],
               loc="center", frameon=False, fontsize=13.5,
               title="Signal category", title_fontsize=14.5)
fig.tight_layout(w_pad=3, h_pad=3.5)
P.save(fig, "Figure_5")

## Key numbers quoted

In [ ]:
print("Leading driver per trait:")
for v in C.MORPHO_VARS:
    top = importance[v].sort_values(ascending=False).head(3)
    print(f"  {C.MORPHO_LABELS[v]:22s} " +
          "  ".join(f"{C.FEATURE_LABELS[f]} {p:.1f}%" for f, p in top.items()))

print("\nMorning R:FR contribution per trait:")
for v in C.MORPHO_VARS:
    print(f"  {C.MORPHO_LABELS[v]:22s} {importance[v]['RFR_morning']:.1f}%")

print("\nSex importance on trait intensity:")
for v in C.MORPHO_VARS:
    print(f"  {C.MORPHO_LABELS[v]:22s} {importance[v]['is_male']:.1f}%")

In [ ]:
assert abs(importance["elongation"]["is_cresc"] - 43.9) < 0.15
assert abs(importance["elongation"]["RFR_morning"] - 14.6) < 0.15
assert abs(importance["metamer_emission"]["night_hours"] - 46.2) < 0.15
assert abs(importance["leaf_area_increase"]["RFR_morning"] - 55.0) < 0.15
assert abs(importance["leaf_shed"]["Tmax"] - 27.7) < 0.15
assert 0.32 < r2["metamer_emission"] < 0.34 and r2["leaf_shed"] < 0.10
print("Validated the Figure 5 driver hierarchy and CV R-squared results used in the manuscript.")